# Truth-Aware Generative AI System
**Major Project | Manipal University Jaipur | Ramya Mishra | 229309090**

---
### What this notebook does
Implements a full **Retrieval-Augmented Generation (RAG)** pipeline:
1. Load & chunk documents
2. Embed chunks with `all-MiniLM-L6-v2` (free, local)
3. Store in FAISS vector index (saved to Google Drive)
4. Retrieve top-k relevant chunks for any query
5. Generate grounded answers with `flan-t5-base` (free, local)
6. *(Phase 2)* NLI-based truth verification & confidence scoring

**No API keys required. Everything runs free on Colab.**

---
> Runtime: Runtime → Change runtime type → **T4 GPU** (recommended)

## Cell 1 — Install dependencies

In [ ]:
# Install all required packages
# Run once per Colab session (takes ~2 min)
!pip install -q \
    faiss-cpu \
    sentence-transformers \
    transformers \
    torch \
    pymupdf \
    python-docx \
    accelerate

print('All packages installed.')

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/truth_aware_ai'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive ready. Project folder: {DRIVE_DIR}')

## Cell 3 — Clone / update GitHub repo

In [ ]:
# ===== EDIT THIS =====
GITHUB_USERNAME = 'YOUR_GITHUB_USERNAME'
REPO_NAME       = 'truth-aware-ai'
# =====================

REPO_URL    = f'https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git'
LOCAL_PATH  = f'/content/{REPO_NAME}'

if not os.path.exists(LOCAL_PATH):
    !git clone {REPO_URL} {LOCAL_PATH}
else:
    !cd {LOCAL_PATH} && git pull

%cd {LOCAL_PATH}
print(f'Working directory: {os.getcwd()}')

## Cell 4 — Check GPU

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU found — running on CPU (slower but works)')

## Cell 5 — Document Loader
Load your documents from Google Drive or paste raw text.

In [ ]:
from document_loader import load_multiple, load_and_chunk

# -------------------------------------------------------
# OPTION A: Use your own documents from Google Drive
# Upload PDFs/TXTs to your Drive folder first, then:
# sources = [
#     f'{DRIVE_DIR}/your_paper.pdf',
#     f'{DRIVE_DIR}/your_notes.txt',
# ]
# -------------------------------------------------------

# OPTION B: Demo with raw text (no files needed)
sources = [
    'text:Retrieval-Augmented Generation (RAG) is an AI framework that grounds '
    'LLM outputs in external knowledge. It retrieves relevant documents from a '
    'knowledge base using vector similarity search, then feeds those documents '
    'as context to a language model to generate a grounded, factual response.',

    'text:Hallucination in large language models (LLMs) is the phenomenon where '
    'the model generates confident but factually incorrect or unverifiable text. '
    'It is a major reliability challenge in production AI systems. RAG significantly '
    'reduces hallucinations by anchoring responses to retrieved evidence.',

    'text:FAISS (Facebook AI Similarity Search) is an open-source library for '
    'efficient similarity search over large collections of dense vectors. '
    'It supports both CPU and GPU and can handle millions of vectors efficiently.',

    'text:Natural Language Inference (NLI) is the task of determining whether a '
    'hypothesis is entailed by, contradicted by, or neutral with respect to a '
    'given premise. NLI models are used in RAG systems to verify factual claims.',

    'text:The DeBERTa model by Microsoft achieves state-of-the-art performance on '
    'NLI benchmarks including MNLI and SNLI. The cross-encoder/nli-deberta-v3-small '
    'variant is fast enough to run on CPU and suitable for real-time fact verification.',

    'text:Sentence transformers convert sentences into dense vector representations '
    'that capture semantic meaning. The all-MiniLM-L6-v2 model produces 384-dimensional '
    'embeddings and achieves excellent performance on semantic similarity tasks.',
]

all_chunks = load_multiple(sources)
print(f'\nTotal chunks loaded: {len(all_chunks)}')
print(f'Preview of first chunk:\n  {all_chunks[0].text[:200]}')

## Cell 6 — Build FAISS Index

In [ ]:
from embedder import VectorStore
from config import RETRIEVAL

# Build or reload the vector index
store = VectorStore()

if os.path.exists(RETRIEVAL.index_path):
    print('Found existing index on Drive — loading...')
    store.load()
    print('Loaded. (Run store.build(all_chunks) to rebuild)')
else:
    print('No existing index — building from scratch...')
    store.build(all_chunks)
    store.save()
    print(f'Index saved to Drive: {RETRIEVAL.index_path}')

## Cell 7 — Test Retrieval

In [ ]:
from retriever import Retriever

retriever = Retriever(store)

test_query = 'What is hallucination in LLMs?'
results, context = retriever.retrieve_and_format(test_query)

print(f'Query: {test_query}')
print(f'Retrieved {len(results)} chunks:\n')
for r in results:
    print(f'  Score: {r.score:.4f} | {r.text[:120]}...')
print(f'\nFormatted context (sent to LLM):\n{context[:600]}...')

## Cell 8 — Load LLM Generator
> This cell downloads the model (~1 GB). Run once per session.

In [ ]:
from generator import RAGGenerator

# Loads flan-t5-base by default (set in config.py)
# To use a larger model: edit LLM_MODEL_NAME in config.py
gen = RAGGenerator()
print('Generator ready.')

## Cell 9 — Run Full RAG Pipeline

In [ ]:
from pipeline import RAGPipeline

# Build pipeline from already-loaded store
pipeline_obj = RAGPipeline(store=store)

# Ask a question
question = 'How does RAG reduce hallucinations in language models?'

result = pipeline_obj.query(question, verbose=True)

## Cell 10 — Interactive Q&A Loop

In [ ]:
# Ask as many questions as you like in a loop
questions = [
    'What is FAISS and how does it work?',
    'Explain Natural Language Inference',
    'What is the all-MiniLM-L6-v2 model used for?',
    'How does DeBERTa perform on NLI tasks?',
]

for q in questions:
    print(f'\n{"="*60}')
    result = pipeline_obj.query(q)
    print(f'Q: {q}')
    print(f'A: {result.answer}')
    print(f'Sources used: {len(result.retrieved_chunks)}')

## Cell 11 — Save results & push to GitHub
Run at the end of every session to save your work.

In [ ]:
import json, datetime

# Save last result as JSON log
log_path = f'{DRIVE_DIR}/result_log.json'
with open(log_path, 'w') as f:
    json.dump(result.to_dict(), f, indent=2)
print(f'Result saved to: {log_path}')

# Push code changes to GitHub
# ===== EDIT these =====
GIT_EMAIL = 'your@email.com'
GIT_NAME  = 'Ramya Mishra'
GH_TOKEN  = 'your_github_pat_here'  # github.com → Settings → Developer Settings → PAT
# ======================

COMMIT_MSG = f'feat: RAG pipeline session {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}'

!cd {LOCAL_PATH} && \
    git config user.email "{GIT_EMAIL}" && \
    git config user.name "{GIT_NAME}" && \
    git remote set-url origin https://{GH_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git && \
    git add -A && \
    git commit -m "{COMMIT_MSG}" && \
    git push

print('Pushed to GitHub!')

---
## Phase 2 Preview — NLI Truth Verification
*(Will be implemented in truth_verifier.py)*

```python
from truth_verifier import TruthVerifier

verifier = TruthVerifier()   # loads cross-encoder/nli-deberta-v3-small
verified = verifier.verify(result)

print(f'Confidence score: {verified.confidence_score:.2f}')
print(f'Verdict: {verified.verdict}')   # verified / uncertain / refused
```